# Manuscript cohort EDA — BigQuery template (`manuscript_notebook_v1`)

**PHI discipline:** Use only `research_id` as a patient identifier. Do not display names, MRNs, free-text operative notes, or narrow calendar dates in notebooks destined for sharing. This template summarizes counts and coarse distributions only.

**Defaults:** `MANUSCRIPT_CODE = "M025"`, cohort from frozen legacy dataset `pub_legacy_source_20260416`. Metadata joins live in `pub_workspace`.

In [ ]:
"""1. Setup — BigQuery client, manuscript parameter, paths, PHI-safe repo root."""
from __future__ import annotations

import json
import os
import re
from datetime import datetime, timezone
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display
from google.cloud import bigquery

# --- BigQuery locations (publication project) ---
BQ_PROJECT = "thyroid-canonical-pub-2026"
BQ_DATASET_WORKSPACE = "pub_workspace"
BQ_DATASET_LEGACY = "pub_legacy_source_20260416"

# --- Parameter: fork notebook or edit this line ---
MANUSCRIPT_CODE = "M025"


def find_repo_root() -> Path:
    here = Path.cwd().resolve()
    for cand in [here, *here.parents]:
        if (cand / "AGENTS.md").exists():
            return cand
    return here


REPO_ROOT = find_repo_root()

# Migration SA key (absolute path). Prefer env; fallback to documented migration checkout.
_DEFAULT_SA_KEY = Path(
    "/Users/loganglosser/Desktop/Thyroid Motherduck To GC migration/_creds/thyroid-pub-loader-key.json"
)
if not os.environ.get("GOOGLE_APPLICATION_CREDENTIALS"):
    if _DEFAULT_SA_KEY.is_file():
        os.environ["GOOGLE_APPLICATION_CREDENTIALS"] = str(_DEFAULT_SA_KEY)
    else:
        raise FileNotFoundError(
            "Set GOOGLE_APPLICATION_CREDENTIALS to the migration repo JSON "
            "(_creds/thyroid-pub-loader-key.json), or update _DEFAULT_SA_KEY in this cell."
        )

client = bigquery.Client(project=BQ_PROJECT)


def manuscript_code_to_id(code: str) -> int:
    m = re.match(r"^[Mm](\d{1,3})$", code.strip())
    if not m:
        raise ValueError(f"MANUSCRIPT_CODE must look like M025; got {code!r}")
    return int(m.group(1))


MANUSCRIPT_ID = manuscript_code_to_id(MANUSCRIPT_CODE)
print("REPO_ROOT:", REPO_ROOT)
print("BQ_PROJECT:", BQ_PROJECT)
print("MANUSCRIPT_CODE:", MANUSCRIPT_CODE, "→ manuscript_id", MANUSCRIPT_ID)
print("GOOGLE_APPLICATION_CREDENTIALS set:", bool(os.environ.get("GOOGLE_APPLICATION_CREDENTIALS")))

In [ ]:
"""2. Manuscript metadata — feasibility × dive map (cohort view name)."""
sql_meta = f"""
SELECT
  f.manuscript_id,
  f.title AS title,
  f.status,
  f.gating_issues,
  f.recommended_next_step,
  f.feasibility_color,
  f.candidate_n AS feasibility_candidate_n,
  d.cohort_view_name,
  d.manuscript_title AS dive_manuscript_title,
  d.dive_type,
  d.canonical_version AS dive_canonical_version
FROM `{BQ_PROJECT}.{BQ_DATASET_WORKSPACE}.manuscript_feasibility_v1` f
JOIN `{BQ_PROJECT}.{BQ_DATASET_WORKSPACE}.manuscript_dive_map_v1` d
  ON f.manuscript_id = d.manuscript_id
WHERE f.manuscript_id = @mid
"""
job_config = bigquery.QueryJobConfig(
    query_parameters=[bigquery.ScalarQueryParameter("mid", "INT64", MANUSCRIPT_ID)]
)
meta_df = client.query(sql_meta, job_config=job_config).to_dataframe()
meta_df

In [ ]:
if meta_df.empty:
    raise RuntimeError(
        f"No joined feasibility+dive_map row for manuscript_id={MANUSCRIPT_ID} ({MANUSCRIPT_CODE})."
    )

cohort_view_name = str(meta_df.iloc[0]["cohort_view_name"]).strip()
if not re.match(r"^[A-Za-z][A-Za-z0-9_]*$", cohort_view_name):
    raise ValueError(f"Unexpected cohort_view_name from catalog: {cohort_view_name!r}")

print("cohort_view_name:", cohort_view_name)

In [ ]:
"""3. Load cohort from frozen legacy dataset."""
cohort_sql = f"SELECT * FROM `{BQ_PROJECT}.{BQ_DATASET_LEGACY}.{cohort_view_name}`"
df = client.query(cohort_sql).to_dataframe()
df.shape

In [ ]:
"""4. Cohort summary — counts only (no PHI narratives)."""


def summarize_cohort_phi_safe(frame: pd.DataFrame) -> pd.DataFrame:
    rows: list[dict[str, object]] = [{"metric": "n_rows", "value": len(frame)}]
    if "research_id" in frame.columns:
        rows.append(
            {
                "metric": "n_distinct_research_id",
                "value": int(frame["research_id"].nunique(dropna=True)),
            }
        )

    demo_cols = [
        "sex",
        "gender",
        "demo_sex_final",
    ]
    for col in demo_cols:
        if col in frame.columns:
            vc = frame[col].astype("string").fillna("<NA>").value_counts()
            for k, v in vc.head(20).items():
                rows.append({"metric": f"count:{col}:{k}", "value": int(v)})
            break

    race_cols = ["race", "demo_race_final", "race_ethnicity_combined"]
    for col in race_cols:
        if col in frame.columns:
            vc = frame[col].astype("string").fillna("<NA>").value_counts()
            for k, v in vc.head(25).items():
                rows.append({"metric": f"count:{col}:{k}", "value": int(v)})
            break

    # Year-only ranges from columns that look like calendar years (numeric)
    for col in frame.columns:
        lower = col.lower()
        if "year" not in lower and not lower.endswith("_yr"):
            continue
        ser = pd.to_numeric(frame[col], errors="coerce")
        if ser.notna().sum() == 0:
            continue
        lo, hi = float(ser.min()), float(ser.max())
        if 1900 <= lo <= 2100 and 1900 <= hi <= 2100:
            rows.append({"metric": f"year_span:{col}", "value": f"{int(lo)}–{int(hi)}"})

    # Datetime-like columns → min/max calendar year (no raw dates printed)
    for col in frame.columns:
        if not pd.api.types.is_datetime64_any_dtype(frame[col]):
            continue
        ser = pd.to_datetime(frame[col], utc=True, errors="coerce")
        if ser.notna().sum() == 0:
            continue
        rows.append(
            {
                "metric": f"year_span_dt:{col}",
                "value": f"{ser.dt.year.min()}–{ser.dt.year.max()}",
            }
        )

    return pd.DataFrame(rows)


summary_tbl = summarize_cohort_phi_safe(df)
summary_tbl

In [ ]:
"""5. Missingness heatmap (matplotlib) — saved under studies/<CODE>/figures/."""
MISSING_HEATMAP_COL_CAP = 120

miss_frac = df.isna().mean().sort_values(ascending=False).head(MISSING_HEATMAP_COL_CAP)

fig, ax = plt.subplots(figsize=(max(14, len(miss_frac) * 0.11), 5))
im = ax.imshow(
    miss_frac.values.reshape(1, -1),
    aspect="auto",
    cmap="magma",
    vmin=0,
    vmax=1,
)
ax.set_yticks([0])
ax.set_yticklabels(["fraction_missing"])
ax.set_xticks(range(len(miss_frac)))
ax.set_xticklabels(miss_frac.index, rotation=90, fontsize=7)
ax.set_title(
    f"{MANUSCRIPT_CODE}: missingness (top {len(miss_frac)} columns by NA rate, capped)"
)
fig.colorbar(im, ax=ax, fraction=0.02, pad=0.02, label="fraction NA")
plt.tight_layout()

figures_dir = REPO_ROOT / "studies" / MANUSCRIPT_CODE / "figures"
figures_dir.mkdir(parents=True, exist_ok=True)
fig_path = figures_dir / "missingness_heatmap_v1.png"
fig.savefig(fig_path, dpi=150, bbox_inches="tight")
plt.show()
print("Saved:", fig_path.resolve())

In [ ]:
"""6. Baseline outcome incidence — recurrence_status_final / death_date when present."""


def outcome_summary(frame: pd.DataFrame) -> pd.DataFrame:
    rows: list[dict[str, object]] = []
    rec_cols = [
        "recurrence_status_final",
        "recurrence_final",
        "recurrence_any",
        "any_recurrence_flag",
    ]
    rcol = next((c for c in rec_cols if c in frame.columns), None)
    if rcol:
        vc = frame[rcol].astype("string").fillna("<NA>").value_counts(dropna=False)
        for k, v in vc.items():
            rows.append({"signal": rcol, "bucket": str(k), "n": int(v)})

    death_cols = ["death_date", "date_of_death", "vital_death_date"]
    dcol = next((c for c in death_cols if c in frame.columns), None)
    if dcol:
        non_null = int(frame[dcol].notna().sum())
        null_ct = int(frame[dcol].isna().sum())
        rows.append({"signal": dcol, "bucket": "record_present", "n": non_null})
        rows.append({"signal": dcol, "bucket": "record_absent", "n": null_ct})

    if not rows:
        rows.append(
            {
                "signal": "_none_found",
                "bucket": "No recurrence_status_final / recurrence_any / death_date column",
                "n": 0,
            }
        )
    return pd.DataFrame(rows)


out_df = outcome_summary(df)
display(out_df)

plot_df = out_df[out_df["signal"] != "_none_found"]
fig2, ax2 = plt.subplots(figsize=(10, max(3, 0.35 * len(plot_df))))
if len(plot_df):
    lbl = plot_df["signal"].astype(str) + " | " + plot_df["bucket"].astype(str)
    ax2.barh(lbl, plot_df["n"].astype(int))
    ax2.set_xlabel("count")
    ax2.set_title(f"{MANUSCRIPT_CODE}: baseline outcome incidence (counts)")
else:
    ax2.text(0.5, 0.5, "No outcome columns", ha="center", va="center")
plt.tight_layout()

out_fig = figures_dir / "baseline_outcomes_v1.png"
fig2.savefig(out_fig, dpi=150, bbox_inches="tight")
plt.show()
print("Saved:", out_fig.resolve())

In [ ]:
"""7. Append cohort snapshot (row count + UTC timestamp) — audit trail."""
snap_path = REPO_ROOT / "studies" / MANUSCRIPT_CODE / "cohort_snapshots.jsonl"
snap_path.parent.mkdir(parents=True, exist_ok=True)

n_rid = None
if "research_id" in df.columns:
    n_rid = int(df["research_id"].nunique(dropna=True))

record = {
    "run_timestamp_utc": datetime.now(timezone.utc).isoformat(),
    "manuscript_code": MANUSCRIPT_CODE,
    "manuscript_id": MANUSCRIPT_ID,
    "cohort_view_name": cohort_view_name,
    "bq_legacy_table": f"{BQ_PROJECT}.{BQ_DATASET_LEGACY}.{cohort_view_name}",
    "cohort_row_count": int(len(df)),
    "n_distinct_research_id": n_rid,
}

with open(snap_path, "a", encoding="utf-8") as fh:
    fh.write(json.dumps(record, sort_keys=True) + "\n")

print("Appended snapshot:", snap_path.resolve())
record